In [ ]:
import sys

from corner import corner

sys.path.append('..')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.flow_matching.helpers import choose_device
from src.flow_matching.loader import (
    load_model_info,
    load_trained_model,
    prob_path_from_config,
)
from src.flow_matching.neural_posterior import NeuralPosterior
from src.flow_matching.simulator import Model as BatchedModel
from src.helpers import gen_parameter_labels

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15101905"

# loading the model (found via job_id, or path)
vector_field, classifier, config = load_trained_model(job_id, device, ema=False)

# load probability path used in training
path = prob_path_from_config(config)

# load useful training parameters
model_info = load_model_info(config, path=path, device=device)

In [ ]:
inf_params  = model_info['inf_params']
N           = model_info['N']
vector_dim  = model_info['vector_dim']
burstparams = model_info['burstparams']
mean        = model_info['mean']
std         = model_info['std']
transdimensional = model_info['transdimensional']
YBKG        = config['path']['p_data']['init_params']['model_params']['ybkg']

In [ ]:
# making sure model is valid for magnetar data
if path.p_data.noise != 'gaussian' or YBKG > 0:
    raise ValueError('Model not trained with zero-centered gaussian noise.')

# generate FM posterior conditioned on **observed** counts

## Read observational data sample

In [ ]:
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+241.347_all_data.dat'
# filepath = '../' 'observational_data/FRBs/FRB20190106B_lc.dat'
filepath = '../' 'observational_data/FRBs/FRB20190124F_lc.dat'
# filepath = '../' 'observational_data/FRBs/FRB20190122C_lc.dat'
# filepath = '../' 'observational_data/FRBs/FRB20190501B_lc.dat'
# filepath = '../' 'observational_data/FRBs/FRB20190115B_lc.dat'

if 'FRB' in filepath:
    df = pd.read_csv(filepath, header=None, delimiter=' ')
    observed_counts = torch.tensor(df[0], device=device)
else: # pulsar
    import h5py
    with h5py.File(filepath, "r") as f:
        # Print all root level object names (aka keys) 
        # these can be group or dataset names 
        print(f.keys())
        data = f.get('profile')
        waterfall = np.array(f.get('waterfall'))
        observed_counts = torch.tensor(data, device=device) 

if '20190122C' in filepath:
    observed_counts = torch.roll(observed_counts, shifts=-120).cpu()

plt.plot([i for i in np.arange(0, 1, 0.001)], observed_counts.cpu(), 'k-')
plt.ylabel("SNR", fontsize=20)
plt.xlabel("t (ms)", fontsize=20)
plt.xlim(0, 1)
plt.ylim(bottom=-2.4)
plt.tick_params(labelsize=18)
plt.tight_layout()
plt.savefig("FRB20190124F", transparent=True)
# plt.imshow(waterfall)

In [ ]:
learned_posterior = NeuralPosterior(
    vector_field,
    classifier,
    model_info,
    path,
    device=device
)

In [ ]:
num_samples = 1000
samples_per_batch = 1000

Ns, theta_samples = learned_posterior.sample(num_samples, observed_counts, samples_per_batch, time_steps=200)
p_N = learned_posterior.p_N
Ns = Ns.to('cpu')
theta_samples

# Classifier p(N)

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

# corner plot (for a given N)

In [ ]:
range_ = 1#0.97
N_inf = 6 # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf 
FM_samples = theta_samples[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf

# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
new_dim = N_inf * len(inf_params)
print( FM_samples.view(bs, len(inf_params), N)[:, :, :N_inf].shape)
FM_samples = FM_samples.view(-1, len(inf_params), N)[:, :, :N_inf].reshape(bs, new_dim)  

var_names = gen_parameter_labels(inf_params, N)
var_names = np.array(var_names).reshape(len(inf_params), N)[:, :N_inf].flatten()
fig = corner(FM_samples.cpu().numpy(), labels=var_names, range=[range_ for _ in range(N_inf * len(inf_params))], 
            color='black', label='FM', plot_density=True, plot_datapoints=True, 
            fill_contours=False, plot_contours=True, hist_kwargs={'density':True}, bins=30, smooth=1)

plt.show()

## posterior samples

In [ ]:
# posterior samples that include all sampled component numbers
posterior_param_samples = path.p_data.prior.samples_as_dict(theta_samples)
posterior_curve_samples = BatchedModel(
    device=device, 
    time=torch.linspace(0, 1, observed_counts.shape[0]), burstparams=posterior_param_samples, ybkg=YBKG, ncomp=Ns
        ).get_flux()

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(11, 4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])  # first subplot wider

ax1 = plt.subplot(gs[0])
ax2 = plt.subplot(gs[1])

ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.6,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']

# plot mean curve
mean_profile = torch.mean(posterior_curve_samples[0:num_samples], dim=0).cpu()
std_profile  = torch.std(posterior_curve_samples[0:num_samples], dim=0).cpu()
time_ = np.linspace(0, 1, observed_counts.shape[0])
ax1.plot(time_, mean_profile, color='white', label='mean profile', linewidth=1)

q1 = torch.quantile(posterior_curve_samples, 0.025, dim=0).cpu()
q3 = torch.quantile(posterior_curve_samples, 0.975, dim=0).cpu()
iqr = q3 - q1
ax1.fill_between(time_, q1, q3, color='red', alpha=0.7, label='std profile')

name = filepath.split('/')[-1].split('_')[0]
ax1.set_title(name, fontsize=16)
# plt.ylim(top=8.5)
# ax1.legend(fontsize=12, loc='upper right')
ax1.grid(linestyle='dotted')
ax1.set_ylabel('SNR', fontsize=14)
ax1.set_xlabel('t', fontsize=14)
ax1.set_xlim(0,1)
# ax1.set_ylim(top=11)
ax1.tick_params(axis='both', labelsize=14)
# ax1.set_facecolor('lightgrey')


ax2.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    # color=sample_colors,
    color='lightsteelblue', 
    edgecolor='black',    # color of the bar edge
    linewidth=1,
    label=range(1, 21)
    )
# ax2.legend()
ax2.set_xlabel("$N$", fontsize=14)
ax2.set_ylabel('$p(N\mid y)$', fontsize=14, labelpad=15)
ax2.tick_params(axis='both', labelsize=14)
# ax2.set_ylim(top=0.6)
plt.xlim(0, 20.5)


import matplotlib.ticker as mticker

ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(f'{name}.png', dpi=300)

plt.show()
print(observed_counts.shape)